### Estrutura Geral dos Dados

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from src.utils.summary_utils import col_summary
import ipywidgets as widgets 
from IPython.display import display,Markdown,clear_output

In [2]:
msk = os.path.join('..','data','raw','breast_msk_2018')
#Dados clinicos sobre pacientes
data_clinical_patient = pd.read_csv(os.path.join(msk,'data_clinical_patient.txt'),sep='\t',comment='#')
#Dados clinicos sobre amostras
data_clinical_sample = pd.read_csv(os.path.join(msk,'data_clinical_sample.txt'),sep='\t',comment='#')
#dados cna (copy number alteration)
cna = pd.read_csv(os.path.join(msk,'data_cna.txt'),sep='\t')
#Dados de alterações no número de cópias do DNA mapeados nas coordenadas do genoma humano de referência hg19 (GRCh37).
cna_hg19 = pd.read_csv(os.path.join(msk,'data_cna_hg19.seg'),sep='\t')
#Dados sobre mutações 
data_mutations = pd.read_csv(os.path.join(msk,'data_mutations.txt'),sep = '\t')



In [ ]:
col_summary(data_clinical_patient,name='data_clinical_patient')

#### **Dataset**: data_clinical_patient

In [ ]:
col_summary(data_clinical_sample, name='data_clinical_sample')

#### **Dataset**: data_clinical_sample

In [5]:
col_summary(cna,name='data_cna')

#### **Dataset**: data_cna

In [6]:
col_summary(cna_hg19,name='data_cna_hg19')

#### **Dataset**: data_cna_hg19

In [7]:
col_summary(data_mutations,name = 'data_mutations')

#### **Dataset**: data_mutations

### Análise Exploratória Basica

In [8]:
gene_counts = data_mutations.loc[:,'Hugo_Symbol'].value_counts().reset_index()
total = gene_counts.loc[:,'count'].sum()
gene_percents = gene_counts.copy()
gene_percents['count'] = gene_percents['count'] / total * 100
gene_percents['count'] = gene_percents['count'].astype(float)

top25_label = widgets.HTML('<h3>Top 25 genes com maior número de mutações')
top25_output = widgets.Output()

bottom25_label = widgets.HTML('<h3>Bottom 25 genes com menor número de mutações')
bottom25_output = widgets.Output()

with top25_output:
    sns.barplot(data=gene_counts.head(25),y='Hugo_Symbol',x='count')
    plt.xlabel('Qtd')
    plt.ylabel('Gene')
    plt.show()

with bottom25_output:
    sns.barplot(data=gene_counts.tail(25),y='Hugo_Symbol',x='count')
    plt.xlabel('Qtd')
    plt.ylabel('Gene')
    plt.xticks(range(3))
    plt.show()


display(widgets.HBox([
    widgets.VBox([top25_label,top25_output]),
    widgets.VBox([bottom25_label,bottom25_output])
]))

Alguns genes apresentam um número muito maior de mutações em comparação com os demais, como **PIK3CA** e **TP53**. Por outro lado, vários genes aparecem apenas uma vez, indicando que podem ser considerados raros e possivelmente filtrados para o treinamento do modelo.

In [9]:
samples = 40

mutations = data_mutations.copy()
mutations.loc[:,'Tumor_Sample_Barcode'] = mutations.loc[:,'Tumor_Sample_Barcode'].apply(lambda x:x[0:9])
patient_mutations = mutations.loc[:,['Tumor_Sample_Barcode','Hugo_Symbol']].groupby('Tumor_Sample_Barcode').agg(list)
patient_mutations.reset_index(inplace=True)
patient_rr = data_clinical_patient.loc[:,['PATIENT_ID','DFS_EVENT','VITAL_STATUS']]
patient_mutations.columns = ['PATIENT_ID','GENE']
patient_mutations = patient_mutations.merge(patient_rr,how='left',on='PATIENT_ID')
patient_mutations.loc[:,'MUTATIONS'] = patient_mutations.loc[:,'GENE'].apply(lambda x:len(x))

mutation_count = patient_mutations.loc[:,['MUTATIONS','DFS_EVENT']]
mutation_count = mutation_count.groupby('MUTATIONS').agg(rec = ('DFS_EVENT','sum'),total=('DFS_EVENT','count'))
mutation_count.reset_index(inplace=True)
mutation_count.loc[:,'rate'] = mutation_count.loc[:,'rec']/mutation_count.loc[:,'total']
mutation_count = mutation_count[mutation_count.loc[:,'total']>=samples]

vital_count = patient_mutations.loc[:,['MUTATIONS','VITAL_STATUS']]
vital_count.loc[:,'VITAL_STATUS'] = vital_count.loc[:,'VITAL_STATUS'].map({'Alive':0,'Deceased':1})
vital_count['VITAL_STATUS'] = pd.to_numeric(vital_count['VITAL_STATUS']).astype('int64')

vital_count = vital_count.groupby('MUTATIONS').agg(alive=('VITAL_STATUS','sum'),total=('VITAL_STATUS','count'))
vital_count.reset_index(inplace=True)
vital_count.loc[:,'rate'] = vital_count.loc[:,'alive']/vital_count.loc[:,'total']
vital_count = vital_count[vital_count.loc[:,'total']>=samples]

r_mc = mutation_count.loc[:,'MUTATIONS'].corr(mutation_count.loc[:,'rate'])
r_vc = vital_count.loc[:,'MUTATIONS'].corr(vital_count.loc[:,'rate'])

mc_label = widgets.HTML('<h3>Número de mutações por Taxa de Recorrência | '+f'Pearson: {r_mc:.2f}')
mc_output = widgets.Output()

vc_label = widgets.HTML('<h3>Número de mutações por Taxa de Mortalidade | '+f'Pearson: {r_vc:.2f}')
vc_output = widgets.Output()

with mc_output:
    sns.lmplot(data=mutation_count,x='MUTATIONS',y='rate')
    plt.xticks(range(0,25,2))
    plt.yticks([i*0.05 for i in range(10,21)])
    plt.xlabel('Número de mutações')
    plt.ylabel('Taxa de Recorrência')
    sns.scatterplot(data=mutation_count,x='MUTATIONS',y='rate',hue='total',palette='viridis')
    plt.legend(title='Amostra(Qtd)')
    plt.xticks(range(0,13,1))
    plt.yticks([i*0.05 for i in range(10,21)])
    plt.show()

with vc_output:
    sns.lmplot(data=vital_count,x='MUTATIONS',y='rate')
    plt.xticks(range(0,25,2))
    plt.yticks([i*0.05 for i in range(10,21)])
    plt.xlabel('Número de Mutações')
    plt.ylabel('Taxa de Mortalidade')
    sns.scatterplot(data=vital_count,x='MUTATIONS',y='rate',hue='total',palette='viridis')
    plt.legend(title='Amostra(Qtd)')
    plt.xticks(range(0,11,1))
    plt.yticks([i*0.05 for i in range(2,7)])
    plt.show()

display(widgets.HBox([
    widgets.VBox([mc_label,mc_output]),
    widgets.VBox([vc_label,vc_output]),
]))


Os resultados indicam que existe uma **correlação moderadamente forte** entre o número de mutações e a taxa de recorrência, sugerindo que, em geral, pacientes com maior número de mutações tendem a apresentar uma maior probabilidade de recorrência.

Por outro lado, a **correlação entre o número de mutações e a taxa de mortalidade é moderadamente fraca**, indicando que o aumento no número de mutações não está tão fortemente associado à mortalidade dos pacientes.

É importante ressaltar que os resultados podem se alterar significativamente se forem adicionados dados referentes aos maiores números de mutações, pois esses grupos possuem amostras muito pequenas.

Para os casos em que existem **mais de 40 amostras**, a correlação de Pearson entre o número de mutações e a taxa de recorrência aumenta para **0.96**, indicando uma relação muito forte. Esse valor sugere que o número de mutações é um dado particularmente relevante e pode ser utilizado como uma **variável importante** na construção de um modelo preditivo para recorrência.

In [86]:
age_dist = data_clinical_sample.loc[:,['PATIENT_ID','INVASIVE_CARCINOMA_DX_AGE']]
dfs_event = data_clinical_patient.loc[:,['PATIENT_ID','DFS_EVENT']]
age_dist = age_dist.merge(dfs_event,how='left',on='PATIENT_ID')
age_dist.loc[:,'age_group'] = age_dist.loc[:,'INVASIVE_CARCINOMA_DX_AGE'].map(lambda x:[f'{i}-{i+5}' for i in range(5,101,5) if i>=x][0])

r_by_age = age_dist.loc[:,['age_group','DFS_EVENT']].groupby('age_group').agg(
    dfs_positive = ('DFS_EVENT','sum'),
    total = ('DFS_EVENT','count')
)
r_by_age.reset_index(inplace=True)
r_by_age.loc[:,'rate'] = r_by_age.loc[:,'dfs_positive']/r_by_age.loc[:,'total']
r_by_age.loc[:,'age_group_int'] = r_by_age.index
r_by_age = r_by_age[r_by_age.loc[:,'total'] > 25]
r_by_age.loc[:,'age_mid'] = r_by_age.loc[:,'age_group'].apply(lambda s:int(s.split('-')[0])+2.5)

m_event = data_clinical_patient.loc[:,['PATIENT_ID','VITAL_STATUS']]
m_by_age = age_dist.merge(m_event,how='left',on='PATIENT_ID')
m_by_age['VITAL_STATUS'] = m_by_age.loc[:,'VITAL_STATUS'].map({'Alive':0,'Deceased':1})
m_by_age = m_by_age.loc[:,['age_group','VITAL_STATUS']].groupby('age_group').agg(
    deceased = ('VITAL_STATUS','sum'),
    total = ('VITAL_STATUS','count')
)
m_by_age.reset_index(inplace=True)
m_by_age.loc[:,'rate'] = m_by_age.loc[:,'deceased']/m_by_age.loc[:,'total']
m_by_age.loc[:,'age_group_int'] = m_by_age.index
m_by_age = m_by_age[m_by_age.loc[:,'total']>25]
m_by_age.loc[:,'age_mid'] = m_by_age.loc[:,'age_group'].apply(lambda s:int(s.split('-')[0])+2.5)

corr_ar = r_by_age.loc[:,'age_mid'].corr(r_by_age.loc[:,'rate'])
age_dist_label = widgets.HTML('<h3>Idade x Recorrencia | '+f'Pearson: {corr_ar:.2f}')
age_dist_output = widgets.Output()

corr_am = m_by_age.loc[:,'age_mid'].corr(m_by_age.loc[:,'rate'])
ctypes_label = widgets.HTML('<h3>Idade x Taxa de Mortalidade | '+f'Pearson: {corr_am:.2f}')
ctypes_output = widgets.Output()


with age_dist_output:
    sns.lmplot(data=r_by_age,x='age_group_int',y='rate')
    sns.scatterplot(data=r_by_age,x='age_group_int',y='rate',hue='total')
    plt.xlabel('Idade')
    plt.ylabel('Taxa de Recorrencia')
    plt.xticks(range(1,13),[f'{i*5+25}-{i*5+30}' for i in range(1,13)],rotation=45)
    plt.legend(title='Amostra(Qtd)')
    plt.show()

with ctypes_output:
    sns.lmplot(data=m_by_age, x='age_group_int',y='rate')
    sns.scatterplot(data=m_by_age, x='age_group_int',y='rate',hue='total')
    plt.xlabel('Idade')
    plt.ylabel('Taxa de Mortalidade')
    plt.xticks(range(1,13),[f'{i*5+25}-{i*5+30}' for i in range(1,13)],rotation=45)
    plt.legend(title='Amostra(Qtd)')
    plt.show()

display(widgets.HBox([
    widgets.VBox([age_dist_label,age_dist_output]),
    widgets.VBox([ctypes_label,ctypes_output])
]))



Os dados do dataset indicam uma **forte correlação negativa** entre a idade ao diagnóstico e a taxa de recorrência do câncer de mama, sugerindo que pacientes mais jovens apresentam uma probabilidade significativamente maior de recorrência.

Em contrapartida, a **correlação entre idade e taxa de mortalidade é fraca**, indicando que a idade tem menor efeito aparente sobre o risco de óbito neste conjunto de dados.

> **Observação:** Para o cálculo da taxa de recorrência, tanto nos dois gráficos analisados quanto no gráfico anterior, foi utilizado o `DFS_EVENT`, que registra a ocorrência de eventos relacionados à doença, incluindo recorrência e morte por câncer.